How does the application know Alice is really Alice before it gives her identity to the agent?

## Create a synthetic token

A JWT generally consists of three encoded sections:

`Header.Payload.Signature`

The payload contains claims such as:
- sub: Who the token represents.
- iss: Who issued it.
- aud: Which application it is intended for.
- exp: When it expires.
The signature lets the verifier detect unauthorized modifications and validate the trusted issuer's signing authority.

For this notebook, we'll simulate the identity provider with an ephemeral symmetric key generated in memory. This is a teaching fixture, not our production authentication design.

In [1]:
import secrets
from datetime import datetime, timedelta, timezone

import jwt

# Ephemeral lab-only signing key
SECRET = secrets.token_bytes(32)

ISSUER = "investigation-lab"
AUDIENCE = "investigation-api"

now = datetime.now(timezone.utc)

alice_token = jwt.encode(
    {
        "sub": "E-101",
        "iss": ISSUER,
        "aud": AUDIENCE,
        "iat": now,
        "exp": now + timedelta(minutes=15),
    },
    SECRET,
    algorithm="HS256",
)

print("Synthetic JWT created")

Synthetic JWT created


In [ ]:
# alice_token

## Verify before trusting

In [7]:
def verify_token(token: str) -> dict:
    return jwt.decode(
        token,
        SECRET,
        algorithms=["HS256"],
        issuer=ISSUER,
        audience=AUDIENCE,
        options={
            "require": ["sub", "iss", "aud", "exp", "iat"]
        },
    )


verified_claims = verify_token(alice_token)

print("Authenticated subject:", verified_claims["sub"])

Authenticated subject: E-101


## Test security failures - Run controlled tests

In [8]:
invalid_tokens = {
    "Tampered signature": alice_token[:-1] + (
        "A" if alice_token[-1] != "A" else "B"
    ),
    "Wrong audience": jwt.encode(
        {
            "sub": "E-101",
            "iss": ISSUER,
            "aud": "another-application",
            "exp": now + timedelta(minutes=15),
            "iat": now,
        },
        SECRET,
        algorithm="HS256",
    ),
    "Expired token": jwt.encode(
        {
            "sub": "E-101",
            "iss": ISSUER,
            "aud": AUDIENCE,
            "exp": now - timedelta(minutes=1),
            "iat": now - timedelta(minutes=20),
        },
        SECRET,
        algorithm="HS256",
    ),
}

for name, token in invalid_tokens.items():
    try:
        verify_token(token)
        print(f"{name}: ACCEPTED")
    except jwt.InvalidTokenError:
        print(f"{name}: REJECTED")

Tampered signature: REJECTED
Wrong audience: REJECTED
Expired token: REJECTED
